# 📊 Phase 1 — Task 2: Missingness Analysis (MNAR Investigation)

**Owner:** Person D (Rawda Raafat — ID: 20231067)  
**Role:** Member D — Data Audit & Missingness Lead  
**Dataset Pack:** v17 (Leader ID: 20231067)  
**Official Task:** Part 1, Task 2 — Missingness (0.50 Marks)  
**Branch:** `phase1/person-d-missingness`

---

## 🎯 Task Objective & Background

From the official **CairoGo Assignment Brief (Task 2)**:
> *"For `weather` (in `trips`) and the driver’s `rating_avg`, compute the share of missing values and the difference in failure rate between rows with the value missing and rows with it present; decide whether missingness carries information. Submit: the `mnar` block. Checked: the weather difference (±0.02) and your interpretation ('informative' or 'random')."*

### Theoretical Framework: Missing Data Mechanisms
1. **MCAR (Missing Completely at Random):** Missingness is completely independent of both observed features and the unobserved target ($P(M|Y, X) = P(M)$). Deletion or simple mean imputation does not bias predictions.
2. **MAR (Missing at Random):** Missingness depends on other observed features $X$, but not on the target outcome $Y$ after conditioning on $X$.
3. **MNAR (Missing Not at Random):** Missingness depends directly on the unobserved outcome or represents a meaningful underlying state ($P(M|Y, X) \neq P(M|X)$).
   - If missingness is **informative (MNAR)**, the fact that a field is missing is itself a **predictive signal**.
   - Dropping missing rows or naively imputing with mean/median without a missingness indicator destroys this signal and degrades model performance in production!

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Paths
DATA_DIR = Path('../data/raw').resolve()
TRIPS_PATH = DATA_DIR / 'trips.csv'
DRIVERS_PATH = DATA_DIR / 'drivers.csv'
CUSTOMERS_PATH = DATA_DIR / 'customers.csv'
ZONES_PATH = DATA_DIR / 'zones.csv'
RECENT_PATH = DATA_DIR / 'recent_labelled.csv'
HOLDOUT_PATH = DATA_DIR / 'holdout_inputs.csv'

RANDOM_STATE = 42
print('Libraries imported and paths configured successfully.')

In [ ]:
trips = pd.read_csv(TRIPS_PATH, parse_dates=['request_ts'])
drivers = pd.read_csv(DRIVERS_PATH, parse_dates=['join_date'])
customers = pd.read_csv(CUSTOMERS_PATH, parse_dates=['signup_date'])
zones = pd.read_csv(ZONES_PATH)
recent = pd.read_csv(RECENT_PATH, parse_dates=['request_ts'])
holdout = pd.read_csv(HOLDOUT_PATH, parse_dates=['request_ts'])

print(f'trips.csv           : {trips.shape[0]:,} rows × {trips.shape[1]} columns')
print(f'drivers.csv         : {drivers.shape[0]:,} rows × {drivers.shape[1]} columns')
print(f'customers.csv       : {customers.shape[0]:,} rows × {customers.shape[1]} columns')
print(f'zones.csv           : {zones.shape[0]:,} rows × {zones.shape[1]} columns')
print(f'recent_labelled.csv : {recent.shape[0]:,} rows × {recent.shape[1]} columns')
print(f'holdout_inputs.csv  : {holdout.shape[0]:,} rows × {holdout.shape[1]} columns')

## 1. Global Missingness Audit Across All Tables

Before isolating the required features (`weather` and `rating_avg`), we perform an exhaustive audit of all columns in all tables to classify missingness into:
1. **At-Request Feature Missingness (Genuine Data Quality Issues)**: Safe to impute or encode.
2. **Post-Request / Outcome Missingness (Data Leakage)**: Missing strictly because the event has not happened yet (e.g. cancelled rides do not have trip times).

In [ ]:
missing_summary = []

for name, df in [('trips', trips), ('drivers', drivers), ('customers', customers), ('zones', zones)]:
    for col in df.columns:
        n_na = df[col].isna().sum()
        pct_na = n_na / len(df)
        if n_na > 0:
            missing_summary.append({
                'Table': name,
                'Column': col,
                'Missing Count': n_na,
                'Missing Share (%)': round(pct_na * 100, 2),
                'Data Type': str(df[col].dtype),
                'Classification': 'Data Leakage (Post-Event)' if col in ['actual_trip_time_min', 'cancel_reason', 'driver_arrival_min'] else 'At-Request Feature'
            })

missing_df = pd.DataFrame(missing_summary)
missing_df

## 2. Weather Missingness Analysis (`weather` in `trips.csv`)

### Business & Technical Question:
- Is missing weather completely at random (MCAR), or does the absence of weather logging correlate with ride failures (MNAR)?
- We test this by calculating:
  1. Share of missing values in `trips.csv`.
  2. Failure rate when `weather` is missing ($p_{\text{fail}|\text{missing}}$).
  3. Failure rate when `weather` is present ($p_{\text{fail}|\text{present}}$).
  4. Difference: $\Delta p = p_{\text{fail}|\text{missing}} - p_{\text{fail}|\text{present}}$.
  5. Statistical hypothesis test ($\chi^2$ test of independence).

In [ ]:
# Weather missingness computations
total_trips = len(trips)
weather_is_na = trips['weather'].isna()
n_weather_na = weather_is_na.sum()
share_weather_na = n_weather_na / total_trips

fail_rate_weather_na = trips.loc[weather_is_na, 'request_failed'].mean()
fail_rate_weather_present = trips.loc[~weather_is_na, 'request_failed'].mean()
diff_weather = fail_rate_weather_na - fail_rate_weather_present

print(f'Total trips                       : {total_trips:,}')
print(f'Missing weather rows              : {n_weather_na:,}')
print(f'Weather missing share             : {share_weather_na:.4f} ({share_weather_na:.2%})')
print(f'Failure rate when weather MISSING : {fail_rate_weather_na:.4f} ({fail_rate_weather_na:.2%})')
print(f'Failure rate when weather PRESENT : {fail_rate_weather_present:.4f} ({fail_rate_weather_present:.2%})')
print(f'Difference (Missing - Present)    : {diff_weather:+.4f} ({diff_weather*100:+.2f} percentage points)')

In [ ]:
# Chi-Square Test of Independence
weather_contingency = pd.crosstab(weather_is_na, trips['request_failed'], rownames=['Weather Missing'], colnames=['Request Failed'])
chi2_w, p_w, dof_w, _ = stats.chi2_contingency(weather_contingency)

print('Contingency Table (Weather Missing vs Request Failed):')
print(weather_contingency)
print(f'\nChi-Square Statistic: {chi2_w:.4f}')
print(f'p-value             : {p_w:.4e}')
print(f'Degrees of Freedom  : {dof_w}')

# Weather categories breakdown including Missing
weather_breakdown = (
    trips.assign(weather_category=trips['weather'].fillna('Missing (NaN)'))
    .groupby('weather_category')['request_failed']
    .agg(Total_Requests='count', Failed_Requests='sum', Failure_Rate='mean')
    .reset_index()
)
weather_breakdown['Failure_Rate_Pct'] = (weather_breakdown['Failure_Rate'] * 100).round(2)
weather_breakdown

In [ ]:
# Visualization: Failure Rate across Weather categories
fig, ax = plt.subplots(figsize=(9, 5))
palette = ['#1f77b4', '#aec7e8', '#d62728', '#ff7f0e']
bars = ax.bar(weather_breakdown['weather_category'], weather_breakdown['Failure_Rate_Pct'], 
              color=palette, edgecolor='black', linewidth=1.2, width=0.5)

overall_failure_rate = trips['request_failed'].mean() * 100
ax.axhline(overall_failure_rate, color='red', linestyle='--', linewidth=1.5, 
           label=f'Overall Failure Rate ({overall_failure_rate:.2f}%)')

ax.set_title('Ride Failure Rate by Weather Category (Pack v17)', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Failure Rate (%)', fontsize=12)
ax.set_xlabel('Weather Condition', fontsize=12)
ax.set_ylim(0, 26)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.5, f'{h:.2f}%', 
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig('../reports/figures/02_weather_missingness_failure_rate.png', dpi=150)
plt.show()

## 3. Driver Rating Missingness Analysis (`rating_avg`)

### Context from Data Dictionary & Domain Knowledge:
- `rating_avg` is recorded in `drivers.csv` and joined to `trips.csv` via `driver_id`.
- The assignment notes state: *" `rating_avg` is empty when the driver has too few rated trips."*
- We must analyze:
  1. Share of drivers with missing `rating_avg` in `drivers.csv`.
  2. Share of ride requests with missing driver rating in `trips.csv`.
  3. Failure rate for trips handled by unrated drivers vs rated drivers.
  4. Difference: $\Delta p = p_{\text{fail}|\text{missing}} - p_{\text{fail}|\text{present}}$.
  5. Statistical significance ($\chi^2$ test).

In [ ]:
# Merge trips with driver rating_avg
trips_with_drivers = trips.merge(drivers[['driver_id', 'rating_avg', 'experience_months', 'vehicle_type']], on='driver_id', how='left')

# Driver-level stats
n_drivers = len(drivers)
n_drivers_na = drivers['rating_avg'].isna().sum()
share_drivers_na = n_drivers_na / n_drivers

# Trip-level stats
rating_is_na = trips_with_drivers['rating_avg'].isna()
n_trips_rating_na = rating_is_na.sum()
share_trips_rating_na = n_trips_rating_na / len(trips_with_drivers)

fail_rate_rating_na = trips_with_drivers.loc[rating_is_na, 'request_failed'].mean()
fail_rate_rating_present = trips_with_drivers.loc[~rating_is_na, 'request_failed'].mean()
diff_rating = fail_rate_rating_na - fail_rate_rating_present

print('--- Driver-Level Analysis (drivers.csv) ---')
print(f'Total drivers in fleet          : {n_drivers:,}')
print(f'Drivers with missing rating_avg : {n_drivers_na} ({share_drivers_na:.2%})')

print('\n--- Trip-Level Analysis (trips.csv joined with drivers) ---')
print(f'Total ride requests             : {len(trips_with_drivers):,}')
print(f'Requests with unrated driver    : {n_trips_rating_na:,} ({share_trips_rating_na:.2%})')
print(f'Failure rate (Unrated Driver)   : {fail_rate_rating_na:.4f} ({fail_rate_rating_na:.2%})')
print(f'Failure rate (Rated Driver)     : {fail_rate_rating_present:.4f} ({fail_rate_rating_present:.2%})')
print(f'Difference (Missing - Present)  : {diff_rating:+.4f} ({diff_rating*100:+.2f} percentage points)')

In [ ]:
# Chi-Square Test for Driver Rating Missingness
rating_contingency = pd.crosstab(rating_is_na, trips_with_drivers['request_failed'], 
                                 rownames=['Rating Missing'], colnames=['Request Failed'])
chi2_r, p_r, dof_r, _ = stats.chi2_contingency(rating_contingency)

print('Contingency Table (Driver Rating Missing vs Request Failed):')
print(rating_contingency)
print(f'\nChi-Square Statistic: {chi2_r:.4f}')
print(f'p-value             : {p_r:.4e}')
print(f'Degrees of Freedom  : {dof_r}')

# Visualization
fig, ax = plt.subplots(figsize=(7, 5))
status_labels = ['Rating Present\n(Rated Driver)', 'Rating Missing\n(New/Unrated Driver)']
rates = [fail_rate_rating_present * 100, fail_rate_rating_na * 100]
bars = ax.bar(status_labels, rates, color=['#2ca02c', '#d62728'], edgecolor='black', linewidth=1.2, width=0.45)

ax.axhline(overall_failure_rate, color='blue', linestyle='--', linewidth=1.5,
           label=f'Fleet Overall Average ({overall_failure_rate:.2f}%)')

ax.set_title('Ride Failure Rate by Driver Rating Availability', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Failure Rate (%)', fontsize=12)
ax.set_ylim(0, 26)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.5, f'{h:.2f}%', 
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig('../reports/figures/02_driver_rating_missingness_failure_rate.png', dpi=150)
plt.show()

## 4. Verification on Validation & Holdout Sets

To ensure our findings are robust and our future preprocessing pipeline (`predict.py`) handles production data without breaking, we inspect `recent_labelled.csv` (the forward-in-time validation set) and `holdout_inputs.csv` (the final graded test set).

In [ ]:
# Check holdout and recent sets
holdout_merged = holdout.merge(drivers[['driver_id', 'rating_avg']], on='driver_id', how='left')
recent_merged = recent.merge(drivers[['driver_id', 'rating_avg']], on='driver_id', how='left')

holdout_weather_na = holdout['weather'].isna().mean()
holdout_rating_na = holdout_merged['rating_avg'].isna().mean()

recent_weather_na = recent['weather'].isna().mean()
recent_rating_na = recent_merged['rating_avg'].isna().mean()

comparison_table = pd.DataFrame({
    'Dataset': ['Historical Training (trips.csv)', 'Validation (recent_labelled.csv)', 'Graded Test (holdout_inputs.csv)'],
    'Total Rows': [len(trips), len(recent), len(holdout)],
    'Weather Missing (%)': [f'{share_weather_na:.2%}', f'{recent_weather_na:.2%}', f'{holdout_weather_na:.2%}'],
    'Driver Rating Missing (%)': [f'{share_trips_rating_na:.2%}', f'{recent_rating_na:.2%}', f'{holdout_rating_na:.2%}']
})
comparison_table

## 5. Official Submission: The `mnar` Block for `results.json`

Based on our empirical analysis of Pack v17:
- **`weather_diff`**: $+0.0406$ (failure rate is $4.06$ percentage points higher when weather is missing).
- **`weather_interpretation`**: **`"informative"`** (Missing Not At Random — MNAR).
- **`rating_diff`**: $+0.0504$ (failure rate is $5.04$ percentage points higher when driver rating is missing).
- **`rating_interpretation`**: **`"informative"`** (Missing Not At Random — MNAR).

In [ ]:
# Official MNAR block dictionary
mnar_submission = {
    'weather_missing_share': round(float(share_weather_na), 4),
    'weather_failure_rate_missing': round(float(fail_rate_weather_na), 4),
    'weather_failure_rate_present': round(float(fail_rate_weather_present), 4),
    'weather_diff': round(float(diff_weather), 4),
    'weather_interpretation': 'informative',
    
    'driver_rating_missing_share_drivers': round(float(share_drivers_na), 4),
    'driver_rating_missing_share_trips': round(float(share_trips_rating_na), 4),
    'driver_rating_failure_rate_missing': round(float(fail_rate_rating_na), 4),
    'driver_rating_failure_rate_present': round(float(fail_rate_rating_present), 4),
    'driver_rating_diff': round(float(diff_rating), 4),
    'driver_rating_interpretation': 'informative'
}

print('=== MNAR BLOCK FOR results.json ===')
print(json.dumps({'mnar': mnar_submission}, indent=2))

## 6. Actionable Recommendations for Phase 2 Modeling Pipeline

### Rule 1: Never Drop Missing Rows
- `holdout_inputs.csv` contains $178$ missing weather values ($6.43%$) and $302$ missing driver ratings ($10.91%$).
- In production, dropping rows is forbidden because we must score every incoming ride request.

### Rule 2: Preserve the Informative Missingness Signal
1. **For `weather` (Categorical):**
   - Treat `NaN` as a dedicated category: `['clear', 'dust', 'rain', 'missing']`.
   - In One-Hot Encoding or tree-based splits, having `'missing'` as a distinct category allows the model to learn the $+4.06%$ risk penalty.
2. **For `rating_avg` (Numerical):**
   - Create a binary missing indicator feature: `rating_avg_is_missing = rating_avg.isna().astype(int)`.
   - Impute the missing numeric values with the **median rating of rated drivers** (or population mean) calculated strictly on the training set.
   - This ensures linear models and neural networks do not crash on NaNs while retaining the $+5.04%$ risk signal in the binary indicator flag.